In [2]:
import os
import time
import requests
import polars as pl
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from pathlib import Path

load_dotenv('../.env')

# ── Paths ──────────────────────────────────────────────────────────────────────
RAW_DATA_PATH  = Path("../data/raw")
PROC_DATA_PATH = Path("../data/processed")
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROC_DATA_PATH.mkdir(parents=True, exist_ok=True)

# ── DB Credentials ─────────────────────────────────────────────────────────────
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB   = os.getenv("POSTGRES_DB",   "cinema_intelligence")
POSTGRES_USER = os.getenv("POSTGRES_USER", "cinema_user")
POSTGRES_PASS = os.getenv("POSTGRES_PASSWORD", "cinema_password")

# ── TMDb ───────────────────────────────────────────────────────────────────────
TMDB_API_KEY  = os.getenv("TMDB_API_KEY")
TMDB_BASE_URL = "https://api.themoviedb.org/3"

# ── IMDb Dataset URLs ──────────────────────────────────────────────────────────
IMDB_URLS = {
    "basics":  "https://datasets.imdbws.com/title.basics.tsv.gz",
    "ratings": "https://datasets.imdbws.com/title.ratings.tsv.gz",
    "akas":    "https://datasets.imdbws.com/title.akas.tsv.gz",
}

# ── Indian Language Mapping ────────────────────────────────────────────────────
INDIAN_LANGUAGES = {
    "hi": "Hindi",   "ta": "Tamil",    "te": "Telugu",
    "ml": "Malayalam","kn": "Kannada", "bn": "Bengali",
    "mr": "Marathi", "pa": "Punjabi",  "gu": "Gujarati",
    "or": "Odia",    "as": "Assamese", "ur": "Urdu",
}

TARGET_YEARS = (2021, 2025)

print("✅ Environment ready")
print(f"   TMDB Key  : {'✔ Found' if TMDB_API_KEY else '✘ MISSING — add to .env'}")
print(f"   PostgreSQL: {POSTGRES_USER}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")


✅ Environment ready
   TMDB Key  : ✔ Found
   PostgreSQL: cinema_user@localhost:5432/cinema_intelligence


In [3]:
def download_imdb_datasets():
    for name, url in IMDB_URLS.items():
        dest = RAW_DATA_PATH / f"{name}.tsv.gz"
        if dest.exists():
            mb = dest.stat().st_size / 1_048_576
            print(f"📦 {name:8s} already exists ({mb:,.0f} MB) — skipping")
            continue

        print(f"📥 Downloading {name} ...")
        t0 = time.time()
        resp = requests.get(url, stream=True, timeout=600)
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        done  = 0
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=65_536):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r   {done/total*100:5.1f}%  {done/1_048_576:,.0f}/{total/1_048_576:,.0f} MB", end="")
        print(f"\n✅ {name} saved in {time.time()-t0:.0f}s")

download_imdb_datasets()


📦 basics   already exists (211 MB) — skipping
📦 ratings  already exists (8 MB) — skipping
📦 akas     already exists (473 MB) — skipping


In [4]:
SCAN_OPTS = dict(
    separator='\t',
    quote_char=None,       # treats " as plain text (critical for movie titles)
    null_values='\\N',     # IMDb's sentinel for missing values
    ignore_errors=True,
    infer_schema_length=10_000,
)

files = {k: RAW_DATA_PATH / f"{k}.tsv.gz" for k in ["basics", "ratings", "akas"]}

basics_lazy  = pl.scan_csv(files["basics"],  **SCAN_OPTS)
ratings_lazy = pl.scan_csv(files["ratings"], **SCAN_OPTS)
akas_lazy    = pl.scan_csv(files["akas"],    **SCAN_OPTS)

print("✅ LazyFrames created (no data loaded yet)")
print("\n--- Basics schema ---")
print(basics_lazy.collect_schema())
print("\n--- Ratings schema ---")
print(ratings_lazy.collect_schema())
print("\n--- AKAs schema ---")
print(akas_lazy.collect_schema())


✅ LazyFrames created (no data loaded yet)

--- Basics schema ---
Schema({'tconst': String, 'titleType': String, 'primaryTitle': String, 'originalTitle': String, 'isAdult': Int64, 'startYear': Int64, 'endYear': String, 'runtimeMinutes': Int64, 'genres': String})

--- Ratings schema ---
Schema({'tconst': String, 'averageRating': Float64, 'numVotes': Int64})

--- AKAs schema ---
Schema({'titleId': String, 'ordering': Int64, 'title': String, 'region': String, 'language': String, 'types': String, 'attributes': String, 'isOriginalTitle': Int64})


In [5]:
# Filter basics: only 'movie' type, years 2021-2025
movies_base = (
    basics_lazy
    .filter(
        (pl.col("titleType") == "movie") &
        (pl.col("startYear").cast(pl.Int32, strict=False)
           .is_between(TARGET_YEARS[0], TARGET_YEARS[1]))
    )
    .select(["tconst", "primaryTitle", "originalTitle",
             "startYear", "runtimeMinutes", "genres"])
)

# Join with ratings
movies_with_ratings = (
    movies_base
    .join(
        ratings_lazy.select(["tconst", "averageRating", "numVotes"]),
        on="tconst",
        how="left"
    )
)

# Quick count
count = movies_with_ratings.select(pl.len()).collect().item()
print(f"✅ Movies found (2021–2025): {count:,}")
print("\n--- Preview ---")
print(movies_with_ratings.limit(5).collect())


✅ Movies found (2021–2025): 102,842

--- Preview ---
shape: (5, 8)
┌───────────┬────────────┬────────────┬───────────┬────────────┬────────────┬───────────┬──────────┐
│ tconst    ┆ primaryTit ┆ originalTi ┆ startYear ┆ runtimeMin ┆ genres     ┆ averageRa ┆ numVotes │
│ ---       ┆ le         ┆ tle        ┆ ---       ┆ utes       ┆ ---        ┆ ting      ┆ ---      │
│ str       ┆ ---        ┆ ---        ┆ i64       ┆ ---        ┆ str        ┆ ---       ┆ i64      │
│           ┆ str        ┆ str        ┆           ┆ i64        ┆            ┆ f64       ┆          │
╞═══════════╪════════════╪════════════╪═══════════╪════════════╪════════════╪═══════════╪══════════╡
│ tt0070596 ┆ Socialist  ┆ El         ┆ 2023      ┆ 78         ┆ Drama,Hist ┆ 6.8       ┆ 238      │
│           ┆ Realism    ┆ realismo   ┆           ┆            ┆ ory        ┆           ┆          │
│           ┆            ┆ socialista ┆           ┆            ┆            ┆           ┆          │
│ tt0077684 ┆ Histórias 

In [6]:
# ── Tier 1: Indian language in ANY AKA entry ──────────────────────────────────
# FIX: Dropped isOriginalTitle constraint (language is null there).
#      Scan ALL AKA entries for Indian language. Use group_by+agg instead of
#      unique() so we always pick a non-null language if one exists.

tier1_indian = (
    akas_lazy
    .filter(pl.col("language").is_in(list(INDIAN_LANGUAGES.keys())))
    .select(["titleId", "language"])
    .group_by("titleId")
    .agg(pl.col("language").first().alias("lang_code"))
    .rename({"titleId": "tconst"})
)

# ── Market presence: separate India and US flags ───────────────────────────────
# FIX: Track India and US separately so we can use both as classification signals.

india_presence = (
    akas_lazy
    .filter(pl.col("region") == "IN")
    .select("titleId").unique()
    .rename({"titleId": "tconst"})
    .with_columns(pl.lit(True).alias("has_india"))
)

us_presence = (
    akas_lazy
    .filter(pl.col("region") == "US")
    .select("titleId").unique()
    .rename({"titleId": "tconst"})
    .with_columns(pl.lit(True).alias("has_us"))
)

# ── Build final target list ────────────────────────────────────────────────────
target_list = (
    movies_with_ratings
    # Left-join presence flags so no movies are dropped yet
    .join(india_presence, on="tconst", how="left")
    .join(us_presence,   on="tconst", how="left")

    # Keep only movies with US or India presence
    .filter(
        pl.col("has_india").fill_null(False) | pl.col("has_us").fill_null(False)
    )

    # Tag Indian-language movies (Tier 1)
    .join(tier1_indian, on="tconst", how="left")

    # Fill boolean nulls
    .with_columns([
        pl.col("has_india").fill_null(False),
        pl.col("has_us").fill_null(False),
    ])

    # ── Classification logic ───────────────────────────────────────────────────
    # Priority 1 (strongest): Has Indian language tag in AKAs → Indian
    # Priority 2 (fallback):  Released in India but NOT in US → Indian
    #                         (catches small Indian films with no language tag)
    # Default:                Hollywood (US-only or global English release)
    .with_columns([
        pl.when(pl.col("lang_code").is_not_null())
          .then(pl.lit("Indian"))
          .when(pl.col("has_india") & ~pl.col("has_us"))
          .then(pl.lit("Indian"))
          .otherwise(pl.lit("Hollywood"))
          .alias("market_type"),

        # FIX: Use when/then instead of replace(..., default=) — avoids deprecation
        pl.when(pl.col("lang_code").is_not_null())
          .then(pl.col("lang_code").replace(INDIAN_LANGUAGES))
          .otherwise(pl.lit("English"))
          .alias("language_name"),
    ])

    .drop(["lang_code", "has_india", "has_us"])
    .unique(subset=["tconst"])   # safe now — classification is done before this
    .collect()
)

print(f"✅ Target list: {target_list.height:,} movies")
print(f"   Hollywood : {target_list.filter(pl.col('market_type')=='Hollywood').height:,}")
print(f"   Indian    : {target_list.filter(pl.col('market_type')=='Indian').height:,}")

print("\n--- Sample Indian titles ---")
print(
    target_list
    .filter(pl.col("market_type") == "Indian")
    .select(["tconst", "primaryTitle", "originalTitle", "language_name"])
    .head(10)
)

print("\n--- Sample Hollywood titles ---")
print(
    target_list
    .filter(pl.col("market_type") == "Hollywood")
    .select(["tconst", "primaryTitle"])
    .head(10)
)


✅ Target list: 59,905 movies
   Hollywood : 48,595
   Indian    : 11,310

--- Sample Indian titles ---
shape: (10, 4)
┌────────────┬──────────────────────────┬──────────────────────────┬───────────────┐
│ tconst     ┆ primaryTitle             ┆ originalTitle            ┆ language_name │
│ ---        ┆ ---                      ┆ ---                      ┆ ---           │
│ str        ┆ str                      ┆ str                      ┆ str           │
╞════════════╪══════════════════════════╪══════════════════════════╪═══════════════╡
│ tt7720142  ┆ Assassin Club            ┆ Assassin Club            ┆ Hindi         │
│ tt28692097 ┆ Nene Naa                 ┆ Nene Naa                 ┆ Hindi         │
│ tt32741121 ┆ Kadaloora Kanmmani       ┆ Kadaloora Kanmmani       ┆ English       │
│ tt13912926 ┆ Sthaayi                  ┆ Sthaayi                  ┆ Hindi         │
│ tt10415600 ┆ Salaga                   ┆ Salaga                   ┆ Hindi         │
│ tt22806134 ┆ The Phenomenon   

In [7]:
output_path = PROC_DATA_PATH / "target_list.parquet"
target_list.write_parquet(output_path)
print(f"✅ Saved to {output_path}")
print(f"   Shape : {target_list.shape}")
print(f"   Size  : {output_path.stat().st_size / 1024:.1f} KB")


✅ Saved to ..\data\processed\target_list.parquet
   Shape : (59905, 10)
   Size  : 1819.8 KB


In [8]:
def get_db_conn():
    return psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASS,
    )

# Test connection
try:
    conn = get_db_conn()
    cur  = conn.cursor()
    cur.execute("SELECT version();")
    print(f"✅ Connected: {cur.fetchone()[0][:50]}")
    conn.close()
except Exception as e:
    print(f"✘ Connection failed: {e}")


✅ Connected: PostgreSQL 16.13 (Debian 16.13-1.pgdg13+1) on x86_


In [9]:
def ingest_languages_and_genres(df: pl.DataFrame):
    conn = get_db_conn()
    cur  = conn.cursor()

    # ── Languages ──────────────────────────────────────────────────────────────
    langs = df["language_name"].unique().to_list()
    execute_values(cur,
        "INSERT INTO languages (language_name) VALUES %s ON CONFLICT DO NOTHING",
        [(l,) for l in langs]
    )

    # ── Genres ────────────────────────────────────────────────────────────────
    all_genres = set()
    for g_str in df["genres"].drop_nulls().to_list():
        all_genres.update(g_str.split(","))
    execute_values(cur,
        "INSERT INTO genres (genre_name) VALUES %s ON CONFLICT DO NOTHING",
        [(g.strip(),) for g in all_genres if g.strip()]
    )

    conn.commit()
    conn.close()
    print(f"✅ Inserted {len(langs)} languages, {len(all_genres)} genres")

ingest_languages_and_genres(target_list)

✅ Inserted 11 languages, 26 genres


In [10]:
def ingest_movies(df: pl.DataFrame):
    conn = get_db_conn()
    cur  = conn.cursor()

    # Fetch language_id lookup
    cur.execute("SELECT language_name, language_id FROM languages")
    lang_map = dict(cur.fetchall())

    # Fetch genre_id lookup
    cur.execute("SELECT genre_name, genre_id FROM genres")
    genre_map = dict(cur.fetchall())

    inserted = 0
    for row in df.iter_rows(named=True):
        # Resolve release_date from startYear
        release_date = f"{row['startYear']}-01-01" if row["startYear"] else None

        # Parse runtime
        try:
            runtime = int(row["runtimeMinutes"]) if row["runtimeMinutes"] else None
        except (ValueError, TypeError):
            runtime = None

        lang_id = lang_map.get(row["language_name"])

        cur.execute("""
            INSERT INTO movies
                (external_id, title, original_title, release_date,
                 runtime_minutes, market_type, language_id, rating, vote_count)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (external_id) DO NOTHING
            RETURNING movie_id
        """, (
            row["tconst"],
            row["primaryTitle"],
            row["originalTitle"],
            release_date,
            runtime,
            row["market_type"],
            lang_id,
            row["averageRating"],
            row["numVotes"],
        ))

        result = cur.fetchone()
        if result:
            movie_id = result[0]
            inserted += 1
            # Insert genres
            if row["genres"]:
                for g in row["genres"].split(","):
                    g = g.strip()
                    if g in genre_map:
                        cur.execute("""
                            INSERT INTO movie_genres (movie_id, genre_id)
                            VALUES (%s, %s) ON CONFLICT DO NOTHING
                        """, (movie_id, genre_map[g]))

    conn.commit()
    conn.close()
    print(f"✅ Inserted {inserted:,} new movies into DB")

ingest_movies(target_list)


✅ Inserted 59,905 new movies into DB


In [11]:
# ── Sync corrected market_type from target_list → DB ─────────────────────────
# Runs in seconds. Fixes all 59,905 movies in the DB using the new target_list.
# TMDb enrichment data (synopsis/financials) is untouched.

conn = get_db_conn()
cur  = conn.cursor()

# Step 1: Insert all Indian languages (were missing — only English was in DB)
for lang_name in INDIAN_LANGUAGES.values():
    cur.execute(
        "INSERT INTO languages (language_name) VALUES (%s) ON CONFLICT DO NOTHING",
        (lang_name,)
    )
conn.commit()

# Step 2: Refresh language_id map
cur.execute("SELECT language_name, language_id FROM languages")
lang_map = dict(cur.fetchall())

# Step 3: Build a lookup of external_id → movie_id from DB
cur.execute("SELECT external_id, movie_id FROM movies")
db_movie_map = dict(cur.fetchall())   # {imdb_id: movie_id}

# Step 4: Apply corrections
india_updated    = 0
hollywood_kept   = 0

for row in target_list.iter_rows(named=True):
    tconst = row["tconst"]
    if tconst not in db_movie_map:
        continue

    movie_id  = db_movie_map[tconst]
    market    = row["market_type"]
    lang_name = row["language_name"]
    lang_id   = lang_map.get(lang_name)

    cur.execute("""
        UPDATE movies
        SET market_type = %s,
            language_id = %s
        WHERE movie_id = %s
    """, (market, lang_id, movie_id))

    if market == "Indian":
        india_updated += 1
    else:
        hollywood_kept += 1

conn.commit()
conn.close()

print(f"✅ DB sync complete")
print(f"   Indian    updated : {india_updated:,}")
print(f"   Hollywood updated : {hollywood_kept:,}")


✅ DB sync complete
   Indian    updated : 11,310
   Hollywood updated : 48,595


In [12]:
import asyncio
import aiohttp
import json
import nest_asyncio

nest_asyncio.apply()

CHECKPOINT_FILE = PROC_DATA_PATH / "tmdb_checkpoint.json"
MAX_CONCURRENT  = 20
COMMIT_EVERY    = 100

def load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        ids = set(json.loads(CHECKPOINT_FILE.read_text()))
        print(f"📂 Checkpoint: {len(ids):,} movies already enriched — skipping them")
        return ids
    return set()

def save_checkpoint(done: set):
    CHECKPOINT_FILE.write_text(json.dumps(list(done)))

async def fetch_one(session: aiohttp.ClientSession, sem: asyncio.Semaphore, imdb_id: str) -> dict | None:
    async with sem:
        try:
            async with session.get(
                f"{TMDB_BASE_URL}/find/{imdb_id}",
                params={"api_key": TMDB_API_KEY, "external_source": "imdb_id"},
                timeout=aiohttp.ClientTimeout(total=15)
            ) as r:
                if r.status == 429:
                    await asyncio.sleep(10)
                    return None
                if r.status != 200:
                    return None
                results = (await r.json()).get("movie_results", [])
                if not results:
                    return None
                tmdb_id = results[0]["id"]

            async with session.get(
                f"{TMDB_BASE_URL}/movie/{tmdb_id}",
                params={"api_key": TMDB_API_KEY},
                timeout=aiohttp.ClientTimeout(total=15)
            ) as r:
                if r.status != 200:
                    return None
                d = await r.json()

            return {
                "imdb_id":  imdb_id,
                "synopsis": d.get("overview") or None,
                "budget":   d.get("budget")   or None,
                "revenue":  d.get("revenue")  or None,
            }
        except Exception:
            return None

async def enrich_async():
    done_ids = load_checkpoint()

    conn = get_db_conn()
    cur  = conn.cursor()
    cur.execute("SELECT external_id, movie_id FROM movies WHERE synopsis IS NULL")
    pending = [(eid, mid) for eid, mid in cur.fetchall() if eid not in done_ids]
    total   = len(pending)
    print(f"🌐 Movies to enrich: {total:,}")

    sem      = asyncio.Semaphore(MAX_CONCURRENT)
    enriched = 0

    async with aiohttp.ClientSession() as session:
        for start in range(0, total, COMMIT_EVERY):
            batch   = pending[start : start + COMMIT_EVERY]
            tasks   = [fetch_one(session, sem, eid) for eid, _ in batch]
            results = await asyncio.gather(*tasks)

            for (imdb_id, movie_id), detail in zip(batch, results):
                if not detail:
                    continue              # ✅ FIX: only checkpoint successful fetches

                done_ids.add(imdb_id)    # ✅ FIX: moved here, after None check

                cur.execute(
                    "UPDATE movies SET synopsis=%s WHERE movie_id=%s",
                    (detail["synopsis"], movie_id)
                )

                b, r = detail["budget"], detail["revenue"]
                roi  = round((r - b) / b * 100, 2) if b and r and b > 0 else None

                cur.execute("""
                    INSERT INTO financials (movie_id, budget_usd, revenue_usd, currency, roi_percentage)
                    VALUES (%s,%s,%s,%s,%s)
                    ON CONFLICT (movie_id) DO UPDATE
                        SET budget_usd=EXCLUDED.budget_usd,
                            revenue_usd=EXCLUDED.revenue_usd,
                            roi_percentage=EXCLUDED.roi_percentage
                """, (movie_id, b, r, "USD", roi))

                enriched += 1

            conn.commit()
            save_checkpoint(done_ids)

            done_count = start + len(batch)
            print(f"  [{done_count:,}/{total:,}] {done_count/total*100:.1f}%  enriched: {enriched:,}", end="\r")

    conn.close()
    print(f"\n✅ Enrichment complete! {enriched:,} movies updated.")
    print(f"   Checkpoint saved → {CHECKPOINT_FILE}")

# ── Delete poisoned checkpoint before running ──────────────────────────────────
if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    print("🗑️  Old checkpoint cleared — starting fresh")

await enrich_async()


🗑️  Old checkpoint cleared — starting fresh
🌐 Movies to enrich: 59,905
  [59,905/59,905] 100.0%  enriched: 0
✅ Enrichment complete! 0 movies updated.
   Checkpoint saved → ..\data\processed\tmdb_checkpoint.json


In [13]:
conn = get_db_conn()
cur  = conn.cursor()

separator = "═" * 55

print(separator)
print("  🎬 CINEMA INTELLIGENCE — Data Pipeline Summary")
print("  Sprint 2: Data Acquisition Complete")
print(separator)

stats = {
    "Total Movies Ingested"  : "SELECT COUNT(*) FROM movies",
    "Hollywood"              : "SELECT COUNT(*) FROM movies WHERE market_type='Hollywood'",
    "Indian"                 : "SELECT COUNT(*) FROM movies WHERE market_type='Indian'",
    "Unique Languages"       : "SELECT COUNT(*) FROM languages",
    "Unique Genres"          : "SELECT COUNT(*) FROM genres",
    "Movies with Synopsis"   : "SELECT COUNT(*) FROM movies WHERE synopsis IS NOT NULL",
    "Movies with Financials" : "SELECT COUNT(*) FROM financials WHERE budget_usd > 0",
    "Movies with Rating"     : "SELECT COUNT(*) FROM movies WHERE rating IS NOT NULL",
}
for label, q in stats.items():
    cur.execute(q)
    print(f"  {label:<26}: {cur.fetchone()[0]:>8,}")

print(separator)
print("  📅 Year Distribution")
print(separator)
cur.execute("""
    SELECT EXTRACT(YEAR FROM release_date)::INT AS yr,
           COUNT(*) AS total,
           SUM(CASE WHEN market_type='Hollywood' THEN 1 ELSE 0 END) AS hollywood,
           SUM(CASE WHEN market_type='Indian'    THEN 1 ELSE 0 END) AS indian
    FROM movies
    WHERE release_date IS NOT NULL
    GROUP BY yr ORDER BY yr
""")
print(f"  {'Year':<6} {'Total':>8} {'Hollywood':>12} {'Indian':>8}")
print(f"  {'-'*6} {'-'*8} {'-'*12} {'-'*8}")
for row in cur.fetchall():
    print(f"  {row[0]:<6} {row[1]:>8,} {row[2]:>12,} {row[3]:>8,}")

print(separator)
print("  🇮🇳 Indian Language Breakdown")
print(separator)
cur.execute("""
    SELECT l.language_name, COUNT(*) AS cnt
    FROM movies m
    JOIN languages l ON m.language_id = l.language_id
    WHERE m.market_type = 'Indian'
    GROUP BY l.language_name ORDER BY cnt DESC
""")
for row in cur.fetchall():
    bar = "█" * (row[1] // 100)
    print(f"  {row[0]:<14}: {row[1]:>5,}  {bar}")

print(separator)
print("  🎭 Top 10 Genres")
print(separator)
cur.execute("""
    SELECT g.genre_name, COUNT(*) AS cnt
    FROM movie_genres mg
    JOIN genres g ON mg.genre_id = g.genre_id
    GROUP BY g.genre_name ORDER BY cnt DESC LIMIT 10
""")
for row in cur.fetchall():
    print(f"  {row[0]:<18}: {row[1]:>6,}")

print(separator)
print("  ⭐ Rating Statistics")
print(separator)
cur.execute("""
    SELECT market_type,
           ROUND(AVG(rating)::numeric, 2) AS avg_rating,
           ROUND(MIN(rating)::numeric, 1) AS min_rating,
           ROUND(MAX(rating)::numeric, 1) AS max_rating,
           COUNT(rating)                  AS rated_count
    FROM movies GROUP BY market_type
""")
for row in cur.fetchall():
    print(f"  {row[0]:<12} | Avg: {row[1]}  Min: {row[2]}  Max: {row[3]}  Rated: {row[4]:,}")

print(separator)
print("  🎥 Top Rated Indian Movies")
print(separator)
cur.execute("""
    SELECT m.title, l.language_name, m.rating, m.vote_count
    FROM movies m
    JOIN languages l ON m.language_id = l.language_id
    WHERE m.market_type = 'Indian' AND m.rating IS NOT NULL
    ORDER BY m.rating DESC, m.vote_count DESC LIMIT 8
""")
print(f"  {'Title':<40} {'Language':<12} {'Rating':>6} {'Votes':>8}")
print(f"  {'-'*40} {'-'*12} {'-'*6} {'-'*8}")
for row in cur.fetchall():
    print(f"  {row[0][:39]:<40} {row[1]:<12} {row[2]:>6} {row[3]:>8,}")

print(separator)
print("  🎬 Top Rated Hollywood Movies")
print(separator)
cur.execute("""
    SELECT title, rating, vote_count FROM movies
    WHERE market_type = 'Hollywood' AND rating IS NOT NULL
    ORDER BY rating DESC, vote_count DESC LIMIT 8
""")
print(f"  {'Title':<45} {'Rating':>6} {'Votes':>8}")
print(f"  {'-'*45} {'-'*6} {'-'*8}")
for row in cur.fetchall():
    print(f"  {row[0][:44]:<45} {row[1]:>6} {row[2]:>8,}")

print(separator)
print("  ✅ Pipeline Status  : COMPLETE")
print(f"  📁 Parquet backup  : data/processed/target_list.parquet")
print(f"  🗄️  Database        : PostgreSQL — cinema_intelligence")
print(separator)

conn.close()


═══════════════════════════════════════════════════════
  🎬 CINEMA INTELLIGENCE — Data Pipeline Summary
  Sprint 2: Data Acquisition Complete
═══════════════════════════════════════════════════════
  Total Movies Ingested     :   59,905
  Hollywood                 :   48,595
  Indian                    :   11,310
  Unique Languages          :       13
  Unique Genres             :       26
  Movies with Synopsis      :        0
  Movies with Financials    :        0
  Movies with Rating        :   39,421
═══════════════════════════════════════════════════════
  📅 Year Distribution
═══════════════════════════════════════════════════════
  Year      Total    Hollywood   Indian
  ------ -------- ------------ --------
  2021     11,240        8,561    2,679
  2022     12,231        8,601    3,630
  2023     12,067        9,291    2,776
  2024     12,260       10,700    1,560
  2025     12,107       11,442      665
═══════════════════════════════════════════════════════
  🇮🇳 Indian Language